In [6]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

tx_counter = 1

def generate_transaction():
    global tx_counter
    
    tx_id = f"TX{tx_counter:04d}"
    tx_counter += 1
    
    user_id = f"u{random.randint(1, 20):02d}"
    amount = round(random.uniform(5.0, 5000.0), 2)
    store = random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"])
    category = random.choice(["elektronika", "odzież", "żywność", "książki"])
    timestamp = datetime.now().isoformat()
    
    return {
        "tx_id": tx_id,
        "user_id": user_id,
        "amount": amount,
        "store": store,
        "category": category,
        "timestamp": timestamp
    }

while True:
    transaction = generate_transaction()
    producer.send('transactions', value=transaction)
    print(transaction)
    time.sleep(1)

Overwriting producer.py


In [7]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx.get('store')
    amount = tx.get('amount', 0)
        if msg_count % 10 == 0:
            print(f"\n{'Sklep':<15} | {'Liczba':<8} | {'Suma':<10} | {'Średnia':<10}")
            print("-" * 55)
            for s in sorted(store_counts.keys()):
                count = store_counts[s]
                suma = total_amount[s]
                srednia = suma / count
                print(f"{s:<15} | {count:<8} | {suma:<10.2f} | {srednia:<10.2f}")
    
# TWÓJ KOD
# Dla każdej wiadomości:
#   1. store_counts[store] += 1
#   2. total_amount[store] += amount
#   3. Co 10 wiadomości: print tabela

Overwriting consumer_count.py


In [1]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    data = message.value
    if data['amount'] > 3000:
        print(f"ALERT: {data['tx_id']} | {data['amount']} PLN | {data['store']} | {data['category']}")

Writing consumer_filter.py


In [2]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    data = message.value
    amount = data['amount']
    
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
        
    data['risk_level'] = risk_level
    print(data)

Writing consumer_enrich.py


In [3]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    data = message.value
    store = data['store']
    amount = data['amount']
    
    store_counts[store] += 1
    
    if store not in total_amount:
        total_amount[store] = 0.0
    total_amount[store] += amount
    
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"{'Sklep':<12} | {'Liczba':<6} | {'Suma':<10} | {'Średnia':<10}")
        for s, count in store_counts.items():
            suma = total_amount[s]
            srednia = suma / count
            print(f"{s:<12} | {count:<6} | {suma:<10.2f} | {srednia:<10.2f}")
        print("-" * 47)

Overwriting consumer_count.py


In [4]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

category_stats = defaultdict(lambda: {'count': 0, 'sum': 0.0, 'min': float('inf'), 'max': 0.0})
msg_count = 0

for message in consumer:
    data = message.value
    category = data['category']
    amount = data['amount']
    
    stats = category_stats[category]
    stats['count'] += 1
    stats['sum'] += amount
    
    if amount < stats['min']:
        stats['min'] = amount
    if amount > stats['max']:
        stats['max'] = amount
        
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"{'Kategoria':<15} | {'Liczba':<6} | {'Suma':<10} | {'Min':<8} | {'Max':<8}")
        for cat, s in category_stats.items():
            print(f"{cat:<15} | {s['count']:<6} | {s['sum']:<10.2f} | {s['min']:<8.2f} | {s['max']:<8.2f}")
        print("-" * 59)

Writing consumer_stats.py


In [5]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='anomaly_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_transactions = defaultdict(list)

print("Nasłuchuję na anomalie (więcej niż 3 transakcje / 60s)...")

for message in consumer:
    data = message.value
    user_id = data['user_id']
    current_time = datetime.fromisoformat(data['timestamp'])
    
    user_transactions[user_id].append(current_time)
    
    recent_transactions = [t for t in user_transactions[user_id] if (current_time - t).total_seconds() <= 60]
    
    user_transactions[user_id] = recent_transactions
    
    if len(recent_transactions) > 3:
        print(f"ALERT! Użytkownik {user_id} wykonał {len(recent_transactions)} transakcji w ciągu ostatnich 60 sekund!")
        

Writing consumer_anomaly.py
